# Day 3 Coding Lab: Identification and Estimation of Causal Effects

This notebook illustrates the Day 3 workflow:

```text
causal question
→ causal estimand
→ identification assumptions
→ identifying functional
→ estimator
→ diagnostics and uncertainty
```

We simulate data from a known structural causal model and compare:

- the naive associational contrast;
- outcome-regression standardization;
- inverse probability weighting;
- augmented inverse probability weighting.

The target estimand is the average treatment effect:

\[
\psi = \mathbb{E}[Y^1 - Y^0].
\]


## 1. Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import expit
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import KFold


## 2. Simulate data from a structural causal model

We generate covariates \(W=(W_1,W_2,W_3)\), a binary treatment \(A\), and a continuous outcome \(Y\).

Treatment is confounded because the same covariates affect both \(A\) and \(Y\).


In [ ]:
def simulate_data(n, seed=1):
    rng = np.random.default_rng(seed)

    w1 = rng.normal(0, 1, n)
    w2 = rng.binomial(1, 0.5, n)
    w3 = rng.normal(0, 1, n)

    # Propensity score: P(A=1 | W)
    pi = expit(-0.2 + 0.8*w1 - 0.7*w2 + 0.5*w3)

    a = rng.binomial(1, pi, n)

    # Heterogeneous treatment effect
    tau = 2.0 + 0.5*w1

    # Baseline outcome regression
    mu0 = 1.0 + w1 + 0.5*w2 - 0.5*w3 + 0.5*w1*w2

    y = mu0 + tau*a + rng.normal(0, 1, n)

    return pd.DataFrame({
        "W1": w1,
        "W2": w2,
        "W3": w3,
        "pi_true": pi,
        "A": a,
        "Y": y,
        "tau_true": tau,
        "mu0_true": mu0
    })


df = simulate_data(3000, seed=123)
df.head()


## 3. True causal effect

Because we know the data-generating mechanism, the true conditional treatment effect is

\[
\tau(W) = 2 + 0.5 W_1.
\]

Therefore, the true ATE is

\[
\psi = \mathbb{E}\{\tau(W)\}.
\]


In [ ]:
true_ate = df["tau_true"].mean()
true_ate


## 4. Naive associational contrast

The naive difference in observed means is

\[
\mathbb{E}(Y \mid A=1) - \mathbb{E}(Y \mid A=0).
\]

This is generally not equal to the causal effect when treatment is confounded.


In [ ]:
naive = df.loc[df["A"] == 1, "Y"].mean() - df.loc[df["A"] == 0, "Y"].mean()

pd.DataFrame({
    "Quantity": ["True ATE", "Naive association"],
    "Estimate": [true_ate, naive],
    "Bias relative to true ATE": [0.0, naive - true_ate]
})


## 5. Outcome-regression standardization

We estimate

\[
Q(a,w)=\mathbb{E}(Y\mid A=a,W=w)
\]

and plug it into the adjustment formula:

\[
\widehat\psi_{\mathrm{OR}}
=
\frac{1}{n}\sum_{i=1}^n
\{\widehat Q(1,W_i)-\widehat Q(0,W_i)\}.
\]


In [ ]:
covariates = ["W1", "W2", "W3"]
X = df[["A"] + covariates]
y = df["Y"]

# A simple linear model with main terms only
outcome_model = LinearRegression()
outcome_model.fit(X, y)

X1 = df[["A"] + covariates].copy()
X0 = df[["A"] + covariates].copy()
X1["A"] = 1
X0["A"] = 0

q1_hat = outcome_model.predict(X1)
q0_hat = outcome_model.predict(X0)

ate_or = np.mean(q1_hat - q0_hat)
ate_or


## 6. Propensity score and inverse probability weighting

We estimate

\[
\pi(w)=\mathbb{P}(A=1\mid W=w)
\]

and compute

\[
\widehat\psi_{\mathrm{IPW}}
=
\frac{1}{n}
\sum_{i=1}^n
\left[
\frac{A_iY_i}{\widehat\pi(W_i)}
-
\frac{(1-A_i)Y_i}{1-\widehat\pi(W_i)}
\right].
\]


In [ ]:
ps_model = LogisticRegression(max_iter=1000)
ps_model.fit(df[covariates], df["A"])

pi_hat = ps_model.predict_proba(df[covariates])[:, 1]
pi_hat = np.clip(pi_hat, 0.01, 0.99)

a = df["A"].to_numpy()
yy = df["Y"].to_numpy()

ate_ipw = np.mean(a * yy / pi_hat - (1 - a) * yy / (1 - pi_hat))
ate_ipw


## 7. Overlap diagnostics

Positivity requires treated and untreated units to have overlapping covariate distributions.

A practical diagnostic is the distribution of estimated propensity scores by treatment group.


In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(pi_hat[df["A"] == 1], bins=30, alpha=0.6, density=True, label="A=1")
plt.hist(pi_hat[df["A"] == 0], bins=30, alpha=0.6, density=True, label="A=0")
plt.xlabel("Estimated propensity score")
plt.ylabel("Density")
plt.title("Overlap diagnostic")
plt.legend()
plt.show()


In [ ]:
weights = a / pi_hat + (1 - a) / (1 - pi_hat)

pd.Series(weights).describe()


## 8. Augmented inverse probability weighting

The AIPW estimator combines outcome regression and propensity-score modelling:

\[
\widehat\psi_{\mathrm{AIPW}}
=
\frac{1}{n}
\sum_{i=1}^n
\left[
\widehat Q(1,W_i)
-
\widehat Q(0,W_i)
+
\frac{A_i}{\widehat\pi(W_i)}
\{Y_i-\widehat Q(1,W_i)\}
-
\frac{1-A_i}{1-\widehat\pi(W_i)}
\{Y_i-\widehat Q(0,W_i)\}
\right].
\]


In [ ]:
aipw_scores = (
    q1_hat - q0_hat
    + a / pi_hat * (yy - q1_hat)
    - (1 - a) / (1 - pi_hat) * (yy - q0_hat)
)

ate_aipw = np.mean(aipw_scores)
se_aipw = np.std(aipw_scores, ddof=1) / np.sqrt(len(df))

ate_aipw, se_aipw


## 9. Compare estimators


In [ ]:
results = pd.DataFrame({
    "Estimator": [
        "True ATE",
        "Naive association",
        "Outcome regression",
        "IPW",
        "AIPW"
    ],
    "Estimate": [
        true_ate,
        naive,
        ate_or,
        ate_ipw,
        ate_aipw
    ]
})

results["Bias relative to true ATE"] = results["Estimate"] - true_ate
results


In [ ]:
plt.figure(figsize=(7, 4))
plt.axhline(true_ate, linestyle="--", label="True ATE")
plt.scatter(results["Estimator"], results["Estimate"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Estimate")
plt.title("Comparison of causal effect estimators")
plt.legend()
plt.tight_layout()
plt.show()


## 10. Optional: cross-fitted AIPW with flexible nuisance models

Cross-fitting helps reduce overfitting bias when flexible machine-learning methods are used for nuisance estimation.

The idea is:

1. Split the sample into folds.
2. Estimate nuisance functions on training folds.
3. Predict nuisance functions on held-out folds.
4. Construct the AIPW estimator using out-of-fold predictions.


In [ ]:
def crossfit_aipw(df, covariates, n_splits=5, seed=123):
    n = len(df)
    q1 = np.zeros(n)
    q0 = np.zeros(n)
    pi = np.zeros(n)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for train_idx, test_idx in kf.split(df):
        train = df.iloc[train_idx]
        test = df.iloc[test_idx]

        # Flexible outcome model
        q_model = RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=10,
            random_state=seed
        )
        q_model.fit(train[["A"] + covariates], train["Y"])

        test1 = test[["A"] + covariates].copy()
        test0 = test[["A"] + covariates].copy()
        test1["A"] = 1
        test0["A"] = 0

        q1[test_idx] = q_model.predict(test1)
        q0[test_idx] = q_model.predict(test0)

        # Flexible propensity model
        g_model = RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=10,
            random_state=seed
        )
        g_model.fit(train[covariates], train["A"])

        pi[test_idx] = g_model.predict_proba(test[covariates])[:, 1]

    pi = np.clip(pi, 0.01, 0.99)
    a = df["A"].to_numpy()
    y = df["Y"].to_numpy()

    scores = (
        q1 - q0
        + a / pi * (y - q1)
        - (1 - a) / (1 - pi) * (y - q0)
    )

    estimate = np.mean(scores)
    se = np.std(scores, ddof=1) / np.sqrt(n)

    return estimate, se, q1, q0, pi, scores


ate_cf_aipw, se_cf_aipw, q1_cf, q0_cf, pi_cf, scores_cf = crossfit_aipw(df, covariates)

ate_cf_aipw, se_cf_aipw


In [ ]:
results2 = pd.concat([
    results,
    pd.DataFrame({
        "Estimator": ["Cross-fitted AIPW"],
        "Estimate": [ate_cf_aipw],
        "Bias relative to true ATE": [ate_cf_aipw - true_ate]
    })
], ignore_index=True)

results2


## 11. Exercises

1. Increase the sample size. What happens to the estimators?
2. Modify the treatment assignment mechanism to create weak overlap. What happens to IPW?
3. Add nonlinearities to the outcome model. Which estimators become biased?
4. Remove an important confounder from the adjustment set. What happens?
5. Compare linear nuisance models with random forests.
6. Try truncating propensity scores at 0.05 and 0.95. How does this affect bias and variance?


## 12. Takeaway

Identification is not estimation.

The adjustment formula tells us what statistical functional equals the causal estimand under assumptions. Estimators approximate that functional from finite samples. Different estimators can behave very differently under model misspecification, weak overlap, and finite-sample noise.
